# 02 · Retrieval & answers
Needs `rag-backend/.env` filled in and `python -m bean_rag.ingest` already run.
Checks which chunks pgvector returns for a question, then the final LLM answer.

In [1]:
from bean_rag.config import settings
from bean_rag.vectorstore import get_client, get_vectorstore

settings.require('google_api_key', 'supabase_url', 'supabase_key')
rows = get_client().table(settings.table_name).select('id', count='exact').limit(1).execute()
print('chunks stored in Supabase:', rows.count)

chunks stored in Supabase: 80


## What does the vector search find?
Higher score = closer meaning (cosine similarity).

In [2]:
store = get_vectorstore()

def show(question, k=settings.top_k):
    print('Q:', question)
    for doc, score in store.similarity_search_with_relevance_scores(question, k=k):
        print(f"  {score:.3f}  {doc.metadata['path']}")
    print()

for q in ['What is your best cold coffee?', 'kya ap delivery karte ho?', 'coffee without milk',
          'what time do you close on sunday', 'cappuccino aur latte me kya farq hai']:
    show(q)

Q: What is your best cold coffee?


  0.716  Menu > Cold Coffee > Iced Latte
  0.715  Frequently Asked Questions > What is your best coffee?
  0.713  Menu > Cold Coffee > Iced Cappuccino
  0.711  Menu > Cold Coffee > Cold Brew

Q: kya ap delivery karte ho?


  0.706  Frequently Asked Questions > Do you deliver?
  0.656  Ordering and Delivery > Delivery
  0.647  Frequently Asked Questions > How much is delivery?
  0.643  Ordering and Delivery > Prices, tax and delivery fee

Q: coffee without milk


  0.743  Frequently Asked Questions > Do you have coffee without milk?
  0.738  Menu > Hot Coffee > Black Coffee
  0.726  Dietary Information > Drinks with and without milk
  0.683  Frequently Asked Questions > What is the strongest coffee?

Q: what time do you close on sunday


  0.712  Frequently Asked Questions > What time do you open and close?
  0.702  Contact and Visit > Opening hours
  0.695  Frequently Asked Questions > Are you open on weekends?
  0.591  Contact and Visit > Phone and email

Q: cappuccino aur latte me kya farq hai


  0.721  Coffee Guide > Cappuccino vs latte vs macchiato
  0.639  Menu > Hot Coffee > Cappuccino
  0.612  Dietary Information > Caffeine guide
  0.604  Dietary Information > Drinks with and without milk



## Full RAG answers

In [3]:
from bean_rag.chain import answer

for q in ['Recommend something sweet and cold', 'How much is a cappuccino with a croissant delivered?',
          'Mujhe allergy hai dairy se, kya order karun?', 'Do you sell pizza?']:
    r = answer(q)
    print('Q:', q)
    print('A:', r.answer)
    print('cards:', r.items, '\n')

Q: Recommend something sweet and cold
A: For something sweet and cold, I recommend the **Mocha Frappe** ($9.80), which is thick, icy, and chocolatey like a coffee dessert! You can find it on the Menu page.
cards: ['Mocha Frappe'] 



Q: How much is a cappuccino with a croissant delivered?
A: A **Cappuccino** and a **Butter Croissant** make a classic breakfast combo! The total comes to $16.80, which includes the subtotal, 10% tax, and $2.50 delivery. You can place your order on the Order Now page!
cards: ['Cappuccino', 'Butter Croissant'] 



Q: Mujhe allergy hai dairy se, kya order karun?
A: Agar aapko dairy se allergy hai, toh aap hamari Menu page par jaakar **Black Coffee**, **Cold Brew**, ya **House Blend** (bina milk ke) try kar sakte hain. Kyunki hamare milk drinks mein dairy hoti hai, isliye order karne se pehle beanscene@mail.com ya +1 202-918-2132 par zaroor contact karein.
cards: ['Black Coffee', 'Cold Brew', 'House Blend'] 



Q: Do you sell pizza?
A: I don't have that information. Please contact Bean Scene at beanscene@mail.com or +1 202-918-2132 for assistance!
cards: [] 



## Follow-up questions use the chat history

In [4]:
history = [{'role': 'user', 'text': 'Tell me about the Mocha Frappe'},
           {'role': 'bot', 'text': 'The Mocha Frappe is a blended chocolate and coffee drink for $9.80.'}]
print(answer('is it served hot?', history).answer)

No, the **Mocha Frappe** is served cold and blended. You can find more details on the Menu page!
